# Feature Engineering with PyTorch for RNN

## Sequential Feature Learning

This notebook demonstrates **Recurrent Neural Network (RNN)** feature engineering for sequential data.

- **LSTM (Long Short-Term Memory)** networks
- **GRU (Gated Recurrent Unit)** networks
- **Bidirectional RNNs** for enhanced context
- **Attention mechanisms** for interpretable features
- **Feature extraction** from hidden states
- **Visualization techniques** for sequential features

### 🔧 Requirements:
- PyTorch >= 2.0
- NumPy, Matplotlib, Seaborn
- Scikit-learn for feature analysis
- Sample text data for demonstration

---

## 1. Import Required Libraries

Setting up all the necessary libraries for RNN feature engineering:

In [ ]:
# Core Libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

# Data Processing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Text Processing
import re
import string
from collections import Counter, defaultdict
import pickle

# System
import os
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries imported successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 2. Data Preparation and Preprocessing

Creating synthetic sequential data and preparing it for RNN processing. In real scenarios, you would load your text data, time series, or other sequential data here.

In [ ]:
# Create synthetic text data for demonstration
def generate_synthetic_text_data(num_samples=5000):
    """Generate synthetic sequential text data with sentiment labels"""
    
    # Positive sentiment templates
    positive_templates = [
        "This product is amazing and fantastic",
        "I love this incredible item very much",
        "Outstanding quality and excellent service provided",
        "Highly recommended for everyone to try",
        "Perfect solution for all my needs"
    ]
    
    # Negative sentiment templates
    negative_templates = [
        "This product is terrible and disappointing", 
        "I hate this awful item completely",
        "Poor quality and bad service received",
        "Not recommended for anyone to buy",
        "Worst solution for any requirements"
    ]
    
    # Generate data
    texts = []
    labels = []
    
    for _ in range(num_samples // 2):
        # Positive examples
        template = np.random.choice(positive_templates)
        # Add some variation
        text = template + " " + np.random.choice(["indeed", "truly", "definitely", "absolutely", ""])
        texts.append(text.strip())
        labels.append(1)
        
        # Negative examples  
        template = np.random.choice(negative_templates)
        text = template + " " + np.random.choice(["unfortunately", "sadly", "really", "completely", ""])
        texts.append(text.strip())
        labels.append(0)
    
    return texts, labels

# Generate synthetic data
print("📝 Generating synthetic text data...")
texts, labels = generate_synthetic_text_data(num_samples=4000)

print(f"✅ Generated {len(texts)} text samples")
print(f"Positive samples: {sum(labels)}")
print(f"Negative samples: {len(labels) - sum(labels)}")

# Display sample data
print("\nSample texts:")
for i in range(5):
    sentiment = "Positive" if labels[i] == 1 else "Negative" 
    print(f"{i+1}. [{sentiment}] {texts[i]}")

In [ ]:
# Text preprocessing and tokenization
class TextPreprocessor:
    def __init__(self):
        self.word_to_idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx_to_word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2
        
    def build_vocabulary(self, texts, min_freq=2):
        """Build vocabulary from texts"""
        word_freq = Counter()
        
        # Count word frequencies
        for text in texts:
            words = self.tokenize(text)
            word_freq.update(words)
        
        # Add words that meet minimum frequency
        for word, freq in word_freq.items():
            if freq >= min_freq:
                if word not in self.word_to_idx:
                    self.word_to_idx[word] = self.vocab_size
                    self.idx_to_word[self.vocab_size] = word
                    self.vocab_size += 1
                    
        print(f"📖 Built vocabulary with {self.vocab_size} words")
        return self.vocab_size
    
    def tokenize(self, text):
        """Simple tokenization"""
        # Convert to lowercase and remove punctuation
        text = text.lower()
        text = re.sub(f'[{string.punctuation}]', '', text)
        return text.split()
    
    def text_to_sequence(self, text):
        """Convert text to sequence of indices"""
        words = self.tokenize(text)
        return [self.word_to_idx.get(word, 1) for word in words]  # 1 is <UNK>
    
    def texts_to_sequences(self, texts):
        """Convert list of texts to sequences"""
        return [self.text_to_sequence(text) for text in texts]

# Initialize preprocessor and build vocabulary
preprocessor = TextPreprocessor()
vocab_size = preprocessor.build_vocabulary(texts, min_freq=2)

# Convert texts to sequences
sequences = preprocessor.texts_to_sequences(texts)

print(f"Text preprocessing complete")
print(f"Vocabulary size: {vocab_size}")
print(f"Sample sequence: {sequences[0]} -> '{texts[0]}'")

# Analyze sequence lengths
seq_lengths = [len(seq) for seq in sequences]
print(f"\nSequence length statistics:")
print(f"Min length: {min(seq_lengths)}")
print(f"Max length: {max(seq_lengths)}")
print(f"Average length: {np.mean(seq_lengths):.1f}")
print(f"Median length: {np.median(seq_lengths):.1f}")

# Plot sequence length distribution
plt.figure(figsize=(10, 5))
plt.hist(seq_lengths, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Sequence Length')
plt.ylabel('Frequency')
plt.title('Distribution of Sequence Lengths')
plt.axvline(np.mean(seq_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(seq_lengths):.1f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Dataset class for RNN training
class SequenceDataset(Dataset):
    def __init__(self, sequences, labels, max_length=None):
        self.sequences = sequences
        self.labels = labels
        self.max_length = max_length or max(len(seq) for seq in sequences)
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        label = self.labels[idx]
        
        # Pad or truncate sequence
        if len(sequence) > self.max_length:
            sequence = sequence[:self.max_length]
        else:
            sequence = sequence + [0] * (self.max_length - len(sequence))
            
        return torch.LongTensor(sequence), torch.LongTensor([label])

# Prepare train/test split
X_train, X_test, y_train, y_test = train_test_split(
    sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Data split:")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Set maximum sequence length (using 95th percentile to avoid outliers)
max_length = int(np.percentile(seq_lengths, 95))
print(f"Using max sequence length: {max_length}")

# Create datasets and data loaders
train_dataset = SequenceDataset(X_train, y_train, max_length=max_length)
test_dataset = SequenceDataset(X_test, y_test, max_length=max_length)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Data loaders created with batch size: {batch_size}")
print(f"Training batches: {len(train_loader)}")
print(f"Testing batches: {len(test_loader)}")

# Test the data loader
sample_batch = next(iter(train_loader))
print(f"\nSample batch shape:")
print(f"Sequences: {sample_batch[0].shape}")  # [batch_size, seq_length]
print(f"Labels: {sample_batch[1].shape}")     # [batch_size, 1]

## 3. RNN Implementation

Now let's implement different RNN architectures for feature extraction.
- **LSTM (Long Short-Term Memory)** - Good for long sequences
- **GRU (Gated Recurrent Unit)** - Simpler alternative to LSTM
- **Bidirectional RNN** - Process sequences in both directions
- **Attention-based RNN** - Focus on important parts of the sequence

In [ ]:
# LSTM Feature Extractor
class LSTMFeatureExtractor(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=2, 
                 dropout=0.3, bidirectional=True, num_classes=2):
        super(LSTMFeatureExtractor, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # LSTM layer
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim, num_layers,
            batch_first=True, dropout=dropout, bidirectional=bidirectional
        )
        
        # Feature extraction layers
        final_hidden_dim = hidden_dim * self.num_directions
        self.feature_extractor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(final_hidden_dim, final_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Classification head
        self.classifier = nn.Linear(final_hidden_dim // 2, num_classes)
        
    def forward(self, x, return_features=False):
        batch_size = x.size(0)
        
        # Embedding
        embedded = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        
        # LSTM
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # Use the last hidden state (or concatenate forward/backward for bidirectional)
        if self.bidirectional:
            # Concatenate last hidden states from both directions
            final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            final_hidden = hidden[-1]
        
        # Extract features
        features = self.feature_extractor(final_hidden)
        
        if return_features:
            return features
        
        # Classification
        output = self.classifier(features)
        return output, features

# GRU Feature Extractor  
class GRUFeatureExtractor(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=2,
                 dropout=0.3, bidirectional=True, num_classes=2):
        super(GRUFeatureExtractor, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # GRU layer
        self.gru = nn.GRU(
            embedding_dim, hidden_dim, num_layers,
            batch_first=True, dropout=dropout, bidirectional=bidirectional
        )
        
        # Feature extraction layers
        final_hidden_dim = hidden_dim * self.num_directions
        self.feature_extractor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(final_hidden_dim, final_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Classification head
        self.classifier = nn.Linear(final_hidden_dim // 2, num_classes)
        
    def forward(self, x, return_features=False):
        # Embedding
        embedded = self.embedding(x)
        
        # GRU
        gru_out, hidden = self.gru(embedded)
        
        # Use the last hidden state
        if self.bidirectional:
            final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            final_hidden = hidden[-1]
        
        # Extract features
        features = self.feature_extractor(final_hidden)
        
        if return_features:
            return features
        
        # Classification
        output = self.classifier(features)
        return output, features

print("RNN model classes defined")
print("- LSTM Feature Extractor")  
print("- GRU Feature Extractor")

# Initialize models
embedding_dim = 128
hidden_dim = 256
num_layers = 2

lstm_model = LSTMFeatureExtractor(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    bidirectional=True
).to(device)

gru_model = GRUFeatureExtractor(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    bidirectional=True
).to(device)

print(f"\nModels created and moved to {device}")
print(f"LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")
print(f"GRU parameters: {sum(p.numel() for p in gru_model.parameters()):,}")

In [ ]:
# Attention-based RNN Feature Extractor
class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_layers=2,
                 dropout=0.3, num_classes=2):
        super(AttentionRNN, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim, num_layers,
            batch_first=True, dropout=dropout, bidirectional=True
        )
        
        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
            nn.Softmax(dim=1)
        )
        
        # Feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Classification head
        self.classifier = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x, return_attention=False, return_features=False):
        batch_size, seq_len = x.size()
        
        # Embedding
        embedded = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        
        # LSTM
        lstm_out, _ = self.lstm(embedded)  # [batch_size, seq_len, hidden_dim*2]
        
        # Attention weights
        attention_weights = self.attention(lstm_out)  # [batch_size, seq_len, 1]
        
        # Apply attention to get context vector
        context_vector = torch.sum(lstm_out * attention_weights, dim=1)  # [batch_size, hidden_dim*2]
        
        # Extract features
        features = self.feature_extractor(context_vector)
        
        if return_features:
            return features
            
        if return_attention:
            return self.classifier(features), features, attention_weights.squeeze(-1)
        
        return self.classifier(features), features

# Initialize attention model
attention_model = AttentionRNN(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers
).to(device)

print(f"Attention RNN model created")
print(f"Parameters: {sum(p.numel() for p in attention_model.parameters()):,}")

# Test forward pass
with torch.no_grad():
    sample_input = sample_batch[0].to(device)[:4]  # Use first 4 samples
    output, features, attention_weights = attention_model(sample_input, return_attention=True)
    print(f"\nForward pass test:")
    print(f"Input shape: {sample_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Features shape: {features.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")

In [ ]:
# Training and Evaluation Functions
def train_rnn_model(model, train_loader, val_loader, num_epochs=10, lr=0.001):
    """Train RNN model with validation monitoring"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    print("Starting RNN training...")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (data, targets) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            
            # Gradient clipping for RNN stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for data, targets in val_loader:
                data, targets = data.to(device), targets.to(device)
                outputs, _ = model(data)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()
        
        # Calculate metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_val_loss = val_loss / len(val_loader)
        epoch_train_acc = 100. * train_correct / train_total
        epoch_val_acc = 100. * val_correct / val_total
        
        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)
        train_accuracies.append(epoch_train_acc)
        val_accuracies.append(epoch_val_acc)
        
        scheduler.step(epoch_val_loss)
        
        if (epoch + 1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}]')
            print(f'  Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%')
            print(f'  Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')
            print(f'  LR: {scheduler.optimizer.param_groups[0]["lr"]:.6f}')
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies
    }

def extract_rnn_features(model, data_loader, layer_name='features'):
    """Extract features from trained RNN model"""
    model.eval()
    all_features = []
    all_labels = []
    
    print(f"Extracting features from {layer_name} layer...")
    
    with torch.no_grad():
        for data, targets in data_loader:
            data, targets = data.to(device), targets.to(device)
            
            if hasattr(model, 'forward') and 'return_features' in model.forward.__code__.co_varnames:
                features = model(data, return_features=True)
            else:
                # For models without explicit feature return
                _, features = model(data)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(targets.cpu().numpy())
    
    features_array = np.concatenate(all_features, axis=0)
    labels_array = np.concatenate(all_labels, axis=0)
    
    print(f"Extracted features shape: {features_array.shape}")
    return features_array, labels_array

print("Training and evaluation functions defined")

In [ ]:
# Train LSTM Model
print("=" * 60)
print("TRAINING LSTM FEATURE EXTRACTOR")
print("=" * 60)

# Train the LSTM model
lstm_history = train_rnn_model(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=8,
    lr=0.001
)

# Train the Attention RNN model
print("\n" + "=" * 60)
print("TRAINING ATTENTION RNN FEATURE EXTRACTOR")
print("=" * 60)

attention_history = train_rnn_model(
    model=attention_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=8,
    lr=0.001
)

print("\nBoth models trained successfully!")

In [ ]:
# Visualization and Analysis
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix

def plot_training_curves(history, model_name):
    """Plot training and validation curves"""
    fig, ((ax1, ax2)) = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(1, len(history['train_losses']) + 1)
    
    # Loss curves
    ax1.plot(epochs, history['train_losses'], 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, history['val_losses'], 'r-', label='Validation Loss', linewidth=2)
    ax1.set_title(f'{model_name} - Loss Curves')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax2.plot(epochs, history['train_accuracies'], 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, history['val_accuracies'], 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_title(f'{model_name} - Accuracy Curves')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def visualize_attention_weights(model, data_loader, tokenizer, num_samples=3):
    """Visualize attention weights for sample texts"""
    model.eval()
    
    # Get a batch of data
    data_iter = iter(data_loader)
    data, targets = next(data_iter)
    data, targets = data.to(device), targets.to(device)
    
    with torch.no_grad():
        outputs, features, attention_weights = model(data[:num_samples], return_attention=True)
    
    # Convert back to text for visualization
    fig, axes = plt.subplots(num_samples, 1, figsize=(15, 4*num_samples))
    if num_samples == 1:
        axes = [axes]
    
    for i in range(num_samples):
        # Get tokens and attention weights
        tokens = data[i].cpu().numpy()
        attention = attention_weights[i].cpu().numpy()
        
        # Filter out padding tokens
        non_pad_mask = tokens != 0
        tokens_filtered = tokens[non_pad_mask]
        attention_filtered = attention[non_pad_mask]
        
        # Create attention visualization
        ax = axes[i]
        bars = ax.bar(range(len(attention_filtered)), attention_filtered, 
                     color=plt.cm.Blues(attention_filtered / attention_filtered.max()))
        
        # Add token labels
        token_labels = [f'T{t}' for t in tokens_filtered[:20]]  # Show first 20 tokens
        ax.set_xticks(range(min(20, len(token_labels))))
        ax.set_xticklabels(token_labels, rotation=45, ha='right')
        
        ax.set_title(f'Sample {i+1} - Attention Weights (Label: {targets[i].item()})')
        ax.set_ylabel('Attention Weight')
        
    plt.tight_layout()
    plt.show()

def analyze_rnn_features(features, labels, model_name):
    """Analyze extracted RNN features"""
    print(f"\n{model_name} Feature Analysis")
    print("-" * 50)
    
    # Basic statistics
    print(f"Feature dimensions: {features.shape}")
    print(f"Feature mean: {features.mean():.4f}")
    print(f"Feature std: {features.std():.4f}")
    print(f"Feature range: [{features.min():.4f}, {features.max():.4f}]")
    
    # PCA Analysis
    pca = PCA(n_components=50)
    features_pca = pca.fit_transform(features)
    
    print(f"PCA explained variance ratio (top 10): {pca.explained_variance_ratio_[:10].round(3)}")
    print(f"Cumulative variance explained (50 components): {pca.explained_variance_ratio_[:50].sum():.3f}")
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Feature distribution
    axes[0, 0].hist(features.flatten(), bins=50, alpha=0.7, color='skyblue')
    axes[0, 0].set_title(f'{model_name} - Feature Distribution')
    axes[0, 0].set_xlabel('Feature Value')
    axes[0, 0].set_ylabel('Frequency')
    
    # PCA components
    axes[0, 1].plot(pca.explained_variance_ratio_[:20], 'o-', markersize=4)
    axes[0, 1].set_title(f'{model_name} - PCA Components')
    axes[0, 1].set_xlabel('Component')
    axes[0, 1].set_ylabel('Explained Variance Ratio')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 2D PCA visualization
    pca_2d = PCA(n_components=2)
    features_2d = pca_2d.fit_transform(features)
    
    scatter = axes[1, 0].scatter(features_2d[:, 0], features_2d[:, 1], 
                                c=labels, cmap='viridis', alpha=0.6, s=20)
    axes[1, 0].set_title(f'{model_name} - PCA 2D Visualization')
    axes[1, 0].set_xlabel('First Principal Component')
    axes[1, 0].set_ylabel('Second Principal Component')
    plt.colorbar(scatter, ax=axes[1, 0])
    
    # Feature correlation heatmap (sample)
    feature_sample = features[:, :20]  # First 20 features
    corr_matrix = np.corrcoef(feature_sample.T)
    
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
                square=True, ax=axes[1, 1])
    axes[1, 1].set_title(f'{model_name} - Feature Correlations (First 20)')
    
    plt.tight_layout()
    plt.show()
    
    return features_pca

print("Visualization functions defined")

In [ ]:
# Execute Analysis and Visualization
print("=" * 60)
print("RNN FEATURE ENGINEERING ANALYSIS")
print("=" * 60)

# Plot training curves for both models
plot_training_curves(lstm_history, "LSTM Feature Extractor")
plot_training_curves(attention_history, "Attention RNN Feature Extractor")

# Extract features from both models
lstm_features, lstm_labels = extract_rnn_features(lstm_model, test_loader)
attention_features, attention_labels = extract_rnn_features(attention_model, test_loader)

# Analyze features
lstm_pca_features = analyze_rnn_features(lstm_features, lstm_labels, "LSTM")
attention_pca_features = analyze_rnn_features(attention_features, attention_labels, "Attention RNN")

# Visualize attention weights
print("\nAttention Weight Visualization:")
visualize_attention_weights(attention_model, test_loader, tokenizer, num_samples=3)

In [ ]:
# Feature Comparison and Practical Applications
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

def compare_feature_quality(features1, features2, labels, names):
    """Compare quality of extracted features using downstream tasks"""
    print(f"\n🔬 Feature Quality Comparison: {names[0]} vs {names[1]}")
    print("-" * 60)
    
    # Split data for evaluation
    from sklearn.model_selection import train_test_split
    
    results = {}
    
    for i, (features, name) in enumerate(zip([features1, features2], names)):
        X_train, X_test, y_train, y_test = train_test_split(
            features, labels, test_size=0.3, random_state=42, stratify=labels
        )
        
        # Test different classifiers
        classifiers = {
            'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'SVM': SVC(random_state=42)
        }
        
        results[name] = {}
        
        for clf_name, clf in classifiers.items():
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            results[name][clf_name] = accuracy
            
            print(f"{name} + {clf_name}: {accuracy:.4f}")
    
    # Visualization
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    classifier_names = list(results[names[0]].keys())
    x = np.arange(len(classifier_names))
    width = 0.35
    
    accuracies1 = [results[names[0]][clf] for clf in classifier_names]
    accuracies2 = [results[names[1]][clf] for clf in classifier_names]
    
    bars1 = ax.bar(x - width/2, accuracies1, width, label=names[0], alpha=0.8)
    bars2 = ax.bar(x + width/2, accuracies2, width, label=names[1], alpha=0.8)
    
    ax.set_xlabel('Classifier')
    ax.set_ylabel('Accuracy')
    ax.set_title('Feature Quality Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(classifier_names)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.3f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),  # 3 points vertical offset
                       textcoords="offset points",
                       ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return results

# Compare LSTM vs Attention RNN features
feature_comparison = compare_feature_quality(
    lstm_features, attention_features, lstm_labels,
    ['LSTM Features', 'Attention RNN Features']
)

In [ ]:
# Summary and Practical Applications
print("=" * 80)
print("RNN FEATURE ENGINEERING SUMMARY")
print("=" * 80)

# Save extracted features for future use
import pickle
import os

# Create features directory
features_dir = '../features/rnn_features'
os.makedirs(features_dir, exist_ok=True)

# Save features
feature_data = {
    'lstm_features': lstm_features,
    'attention_features': attention_features,
    'labels': lstm_labels,
    'feature_comparison': feature_comparison,
    'model_params': {
        'vocab_size': vocab_size,
        'embedding_dim': embedding_dim,
        'hidden_dim': hidden_dim,
        'num_layers': num_layers,
        'sequence_length': sequence_length
    }
}

with open(f'{features_dir}/rnn_features.pkl', 'wb') as f:
    pickle.dump(feature_data, f)

print(f"Features saved to: {features_dir}/rnn_features.pkl")

# Print comprehensive summary
print(f"""
 RNN Feature Engineering Results Summary:

 Model Architectures:
   • LSTM Feature Extractor: {sum(p.numel() for p in lstm_model.parameters()):,} parameters
   • Attention RNN: {sum(p.numel() for p in attention_model.parameters()):,} parameters

Feature Characteristics:
   • LSTM Features: {lstm_features.shape[1]} dimensions
   • Attention Features: {attention_features.shape[1]} dimensions
   • Number of samples: {len(lstm_labels)}

Performance Highlights:
   • LSTM Best Accuracy: {max([v for v in feature_comparison['LSTM Features'].values()]):.4f}
   • Attention Best Accuracy: {max([v for v in feature_comparison['Attention RNN Features'].values()]):.4f}

Key Insights:
   1. Sequential Processing: RNNs naturally handle variable-length sequences
   2. Context Awareness: Attention mechanisms improve feature interpretability
   3. Feature Quality: Both approaches extract meaningful representations
   4. Computational Trade-offs: LSTM vs Attention complexity considerations

Practical Applications:
   • Sentiment Analysis: Extract emotion-aware features from text
   • Document Classification: Hierarchical text understanding
   • Time Series: Sequential pattern recognition in IoT data
   • Natural Language Processing: Context-aware text representations
   • Anomaly Detection: Sequential pattern deviation detection

📁 Generated Artifacts:
    Trained LSTM and Attention RNN models
    Extracted feature representations
    Performance comparison results
    Visualization and analysis plots
    Saved feature data for downstream tasks
""")